In [ ]:
# 이미지 데이터의 기초
# 픽셀(pixel): 이미지를 구성하는 가장 기본 단위. 이미지는 사실 무수한 점(픽셀)들의 집합
# RGB: 빨강(R), 초록(G), 파랑(B) 세 채널로 색을 표현. 각 채널은 0~255 값을 가짐
# (0,0,0) = 검정, (255,255,255) = 흰색
# 이미지 배열 구조: (width, height, color) — 3차원 배열. RGB면 채널 수 3
# 흑백(grayscale): 채널이 1개라 2차원 배열로 표현 (0=검정 ~ 255=흰색)

In [ ]:
# PIL(Pillow) 라이브러리로 이미지 전처리
from PIL import Image

In [ ]:
img = Image.open("../chapter06/Lenna.png")

print(img.size)    # (width, height) 픽셀 수
print(img.format)  # 이미지 포맷 (PNG, JPEG 등)
print(img.mode)    # 색상 모드 (RGB, RGBA 등)

(512, 512)
PNG
RGB


In [16]:
img_resized = img.resize((300, 150))  # (가로, 세로) — 순서쌍 필수
img_resized.size

(300, 150)

In [17]:
img_cropped = img.crop((50, 100, 200, 200))  # (x1, y1, x2, y2)
print(img_cropped.size)  # width=x2-x1, height=y2-y1

(150, 100)


In [18]:
img_gray = img_cropped.convert("L")  # 'L'=그레이스케일, '1'=이진화, 'RGB', 'RGBA' 등

In [19]:
# PIL 이미지 → Numpy 배열 변환 (모델 입력용, 중요)
from PIL import ImageFilter
img_blur = img.filter(ImageFilter.BLUR) 

In [20]:
img_blur.save("Lenna.png")

In [ ]:
from PIL import ImageDraw
# 불러온 이미지 위에 그림을 그리거나 text를 입력할 수 있도록 draw 객체를 생성
draw = ImageDraw.Draw(img)

x = 100
y = 50

# 픽셀 값 직접 읽고 쓰기 (모자이크 등에 사용)
rgb = img.getpixel((x, y))
r, g, b = 255, 0, 0
img.putpixel((x, y), (r, g, b))

In [22]:
# CNN 핵심 3줄
# 문제: MLP는 이미지를 1차원으로 펴서 공간정보를 잃음 → CNN은 2차원 유지
# 해결책: 작은 커널(가중치)이 이미지를 훑으며 특징을 뽑아냄(합성곱) → 이 가중치를 전체 위치에서 공유해서 FC보다 파라미터 수가 훨씬 적음
# 구조: Conv층 여러 개(특징 추출) → Flatten → Dense(fc, 분류) — Flatten은 맨 마지막에만

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, Flatten, Dense
from PIL import Image

# 이미지 불러오기 + CNN 입력 형태로 전처리
img = Image.open("../chapter06/Lenna.png")
img_resized = img.resize((28, 28)).convert("L")   # 28x28 흑백으로 통일
img_array = np.array(img_resized) / 255.0          # 0~1 정규화
img_input = img_array.reshape(1, 28, 28, 1)         # (배치, 높이, 너비, 채널)

# CNN 구조 (Conv 쌓기 → Flatten → Dense)
model = Sequential([
    Conv2D(input_shape=(28,28,1), kernel_size=(3,3), filters=16),
    Conv2D(kernel_size=(3,3), filters=32),
    Conv2D(kernel_size=(3,3), filters=64),
    Flatten(),
    Dense(units=128, activation='relu'),
    Dense(units=10, activation='softmax')
])

# 학습 없이, Lenna 이미지 1장을 모델에 통과시켜 결과 확인
output = model.predict(img_input)

print("입력 shape:", img_input.shape)
print("출력 shape:", output.shape)
print("출력값(10개 클래스 확률):", output)

In [ ]:
# Batch Normalization = 매 층마다 그 층 출력값을 "평균 0, 흩어진 정도 1"로 다시 맞춰주는 작업(γ, β도 이때 함께 계산)
# 왜: 층마다 값의 분포가 들쭉날쭉해지는 걸 막아서 학습을 안정시킴
# γ, β는 정규화 후 "다시 얼마나 펼지, 얼마나 옮길지" 학습하는 파라미터 (fc의 weight/bias와 같은 성격)

In [ ]:
# 데이터 증강 (Data Augmentation)
# 왜: 데이터가 적으면 과적합 위험 → 이미지를 변형(회전, 대칭, 자르기 등)해서 데이터를 강제로 늘림
# 원리: 사람 눈엔 같아 보여도 컴퓨터(모델)는 픽셀이 다른 완전히 새 데이터로 인식 → 그래서 증강 효과가 생김
# Cutout/Mixup/Cutmix는 "더 정교한 증강 기법도 있다" 정도만 알면 됨, 세부 구현은 필요할 때 찾아보면 됨

# 전이 학습 (Transfer Learning) — 예지보전 실무와 직결
# 핵심 아이디어: 데이터 적어도, 대규모로 이미 학습된 모델(ImageNet 등)을 가져와서 일부만 재학습
# 왜 가능한가: CNN 앞쪽 층(선/모서리 같은 저수준 특징)은 어떤 이미지든 공통 → 재사용 가치 있음
# 실무에서 제일 흔한 전략: 나머지는 다 얼리고(freeze), 마지막 classifier만 새로 학습
# 실무 판단 순서:
#     사전학습 모델 불러오기 (weights='imagenet', include_top=False로 classifier 제외)
#     그 모델 전체를 얼림 (trainable = False)
#     내 문제(정상/이상 등)에 맞는 새 classifier만 붙여서 그 부분만 학습

# Style Transfer = 사전학습된 CNN의 feature map을 이용해, 한 이미지의 내용(content)과 다른 이미지의 스타일(style)을 합쳐 새 이미지를 생성하는 기법.
# GAN = Generator(가짜 데이터 생성)와 Discriminator(진짜/가짜 판별)가 서로 경쟁하며 학습해서, 결국 진짜처럼 보이는 새로운 데이터를 만들어내는 프레임워크.

In [ ]:
# 영상 처리 (Image Processing)
# 정의: 입력 영상을 원하는 영상으로 변환하거나, 영상에서 특성(feature)을 추출하는 것
#       → 사람이 이해할 수 있는 형태로 만드는 것

# 1) 저수준 영상 처리 (Low-level)
#    - 입력: 영상 → 출력: 영상
#    - 목적: 사람이 "보기에" 개선됨 (화질 개선, 노이즈 제거, 대비 조정 등)

# 2) 고수준 영상 처리 (High-level)
#    - 입력: 영상 → 출력: 의미 있는 정보 (라벨, 좌표 등)
#    - 목적: 객체/내용을 인식·표현할 수 있음

# 영상의 디지털화 = 표본화 + 양자화
# 표본화(Sampling): 연속된 공간을 일정 간격으로 나눠 이산적인 점(픽셀)으로 만드는 것 → 해상도 결정
# 양자화(Quantization): 각 점의 밝기(연속값)를 정해진 단계의 정수로 바꾸는 것 → 보통 8bit(0~255)

In [7]:
import cv2

# imread: 디스크에 있는 이미지 파일(png, jpg 등)을 읽어서 numpy 배열로 메모리에 올려주는 함수
# 컬러로 읽기 → 채널 3개(B, G, R) 포함된 배열로 반환 (OpenCV는 RGB가 아니라 BGR 순서)
color_img = cv2.imread("Lenna.png", cv2.IMREAD_COLOR)
# 흑백으로 읽기 → 채널 없이 밝기값 1개만 있는 배열로 반환
gray_img = cv2.imread("Lenna.png", cv2.IMREAD_GRAYSCALE)

# (세로 픽셀 수, 가로 픽셀 수, 채널 수) 형태로 출력됨. 예: (512, 512, 3)
print(color_img.shape)
# 채널 정보 없이 (세로, 가로)만 출력됨. 예: (512, 512)
print(gray_img.shape)

# imwrite: 이미지 저장
print("저장 완료") if cv2.imwrite("gray_Lenna.png", gray_img) else print("저장 실패")
# cvtColor: 색깔 바꾸기
gray_img = cv2.cvtColor(color_img, cv2.COLOR_BGR2GRAY)

cv2.imwrite("converted_Lenna.png", gray_img)

# resize: 이미지 크기 변경
resized_img = cv2.resize(color_img, (1024, 1024))
print(resized_img.shape)

cropped_image = color_img[:color_img.shape[0] // 2, :color_img.shape[1] // 2]

print(cropped_image.shape)
cv2.imwrite("cropped_Lenna.png", cropped_image)

(512, 512, 3)
(512, 512)
저장 완료
(1024, 1024, 3)
(256, 256, 3)


True

In [8]:
# rectangle: 이미지 위에 사각형 그리기 (얼굴 부근 박스)
rect_img = color_img.copy()  # rectangle은 원본을 직접 수정하므로 복사본에 그림
cv2.rectangle(rect_img, (220, 200), (400, 420), (0, 255, 0), 2)  # (좌상단), (우하단), 색(BGR), 두께
cv2.imwrite("rect_Lenna.png", rect_img)

True